# Qwen3.5 Multimodal NER — Green Book Entry Extraction

Tests Qwen3.5 vision models as a local, zero-cost alternative to Gemini for entry extraction.
Outputs YAML instead of JSON — small models handle YAML syntax far more reliably.

Inspired by: [Parsing 3.6M Historical Names with Small Models](https://wjbmattingly.com/blog/parsing-3-6-million-historical-names-with-small-models/)

**Qwen3.5 models are natively multimodal** — they read the page scan directly, the same way
Gemini does in `--mode multimodal`. Three extraction modes are supported:

| Mode | What the model sees | When to use |
|------|--------------------|--------------|
| `image_and_ocr` | page scan + OCR transcript | **recommended** — layout context + text accuracy |
| `image_only` | page scan only | tests pure vision; useful if OCR is poor quality |
| `text_only` | OCR transcript only | text-only baseline for comparison |

**GPU requirements:**

| Model | VRAM (bf16) | VRAM (4-bit) | Colab tier |
|-------|------------|--------------|------------|
| `Qwen/Qwen3.5-2B` | ~5 GB | ~2 GB | Free T4 |
| `Qwen/Qwen3.5-4B` | ~9 GB | ~3 GB | Free T4 (4-bit) or L4 |
| `Qwen/Qwen3.5-9B` | ~18 GB | ~6 GB | **L4 (recommended)** |
| `Qwen/Qwen3.5-27B` | ~54 GB | ~18 GB | A100 80 GB |

---
## Setup

In [8]:
%pip install -q --upgrade git+https://github.com/huggingface/transformers.git
%pip install -q accelerate bitsandbytes pyyaml pillow
print('\n✓ Dependencies installed')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

✓ Dependencies installed


In [9]:
from google.colab import drive
drive.mount('/content/drive')

# ── Edit this path to point to your directory-pipeline folder on Drive ────────
PIPELINE_DIR = "/content/drive/Othercomputers/My_Mac/directory-pipeline"
# ─────────────────────────────────────────────────────────────────────────────

import os, sys
os.chdir(PIPELINE_DIR)
print('✓ Working directory:', os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Working directory: /content/drive/Othercomputers/My_Mac/directory-pipeline


In [10]:
from pathlib import Path

# ── Model choice ──────────────────────────────────────────────────────────────
# Qwen3.5-9B fits in bf16 on an L4 (no quantization needed).
# Drop to Qwen3.5-4B for a free T4, or enable USE_4BIT below.
MODEL_ID = 'Qwen/Qwen3.5-9B'

# Set to True to load in 4-bit (enables larger models on smaller GPUs; slight quality drop)
USE_4BIT = False

# ── Extraction mode ───────────────────────────────────────────────────────────
# 'image_and_ocr' : send page scan + OCR text  (recommended)
# 'image_only'    : send page scan only  (model reads the image directly)
# 'text_only'     : send OCR text only   (baseline; no vision tokens)
EXTRACTION_MODE = 'image_and_ocr'

# ── Volume to test ────────────────────────────────────────────────────────────
# Path to a volume directory relative to PIPELINE_DIR.
# Must contain *_gemini*.txt OCR files and a ner_prompt.md.
VOLUME = 'output/green_books_and_related/the_green_book_9ea5d5b0/4bea2040-1118-0132-673e-58d385a7b928'

# How many pages to process (None = all pages)
MAX_PAGES = 10

# Max image size in pixels on longest edge — reduce to 896 if you hit VRAM errors
MAX_IMAGE_PX = 1344

# ─────────────────────────────────────────────────────────────────────────────

vol_path = Path(PIPELINE_DIR) / VOLUME
ocr_files = sorted(vol_path.glob('*_gemini*.txt'))
ner_prompt_path = vol_path / 'ner_prompt.md'

print(f'Volume:       {VOLUME}')
print(f'OCR pages:    {len(ocr_files)}')
print(f'NER prompt:   {"found" if ner_prompt_path.exists() else "NOT FOUND"}')
print(f'Model:        {MODEL_ID}  (4-bit={USE_4BIT})')
print(f'Mode:         {EXTRACTION_MODE}')
print(f'Pages to run: {MAX_PAGES if MAX_PAGES else len(ocr_files)}')

Volume:       output/green_books_and_related/the_green_book_9ea5d5b0/4bea2040-1118-0132-673e-58d385a7b928
OCR pages:    84
NER prompt:   found
Model:        Qwen/Qwen3.5-9B  (4-bit=False)
Mode:         image_and_ocr
Pages to run: 10


---
## Load Model

In [11]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

print(f'Loading {MODEL_ID} ...')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slow)"}')

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

load_kwargs = dict(
    device_map='auto',
    dtype=torch.bfloat16,
    trust_remote_code=True,
)
if USE_4BIT:
    load_kwargs['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()

vram = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
print(f'✓ Model loaded  ({vram:.1f} GB VRAM used)')

Loading Qwen/Qwen3.5-9B ...
GPU: NVIDIA L4


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

✓ Model loaded  (18.9 GB VRAM used)


---
## Locate Page Images

Each OCR file (`0001_5205889_gemini-3.1-flash-lite-preview.txt`) has a corresponding
downloaded page JPEG (`0001_5205889.jpg`) in the same volume directory.
This cell builds the mapping and reports how many images were found.

In [13]:
from PIL import Image as PILImage

def find_page_image(ocr_path: Path) -> Path | None:
    """Return the page JPEG for an OCR file, or None if not found.

    OCR:   0001_5205889_gemini-3.1-flash-lite-preview.txt
    Image: 0001_5205889.jpg
    """
    stem = ocr_path.stem
    # Strip the model slug: everything before the first '_gemini' occurrence
    prefix = stem.split('_gemini')[0] if '_gemini' in stem else '_'.join(stem.split('_')[:2])
    candidates = [
        p for p in sorted(ocr_path.parent.glob(f'{prefix}.jpg'))
        if '_viz' not in p.name
    ]
    return candidates[0] if candidates else None


def load_image(image_path: Path, max_px: int = MAX_IMAGE_PX) -> PILImage.Image:
    """Load and resize a page image if it exceeds max_px on the longest edge."""
    img = PILImage.open(image_path).convert('RGB')
    w, h = img.size
    if max(w, h) > max_px:
        scale = max_px / max(w, h)
        img = img.resize((int(w * scale), int(h * scale)), PILImage.LANCZOS)
    return img


# Build OCR -> image map for the whole volume
image_map: dict[Path, Path | None] = {p: find_page_image(p) for p in ocr_files}
found = sum(1 for v in image_map.values() if v is not None)

print(f'Images found: {found} / {len(ocr_files)}')
if found == 0 and EXTRACTION_MODE != 'text_only':
    print('WARNING: no images found — extraction will fall back to text_only mode')
elif found < len(ocr_files):
    missing = [p.name for p, img in image_map.items() if img is None]
    print(f'Missing ({len(missing)}): {missing[:5]}{"..." if len(missing) > 5 else ""}')
else:
    sample_path = next(v for v in image_map.values() if v is not None)
    w, h = PILImage.open(sample_path).size
    print(f'Sample: {sample_path.name}  {w}x{h} px')

Images found: 84 / 84
Sample: 0001_5207618.jpg  2048x2702 px


---
## Build the YAML Extraction Prompt

We load the volume's existing `ner_prompt.md` as the system prompt and append
YAML-specific output instructions.

In `image_and_ocr` mode the user turn contains the page scan followed by the OCR transcript
labelled as a reference — the model can use the image for layout context and the transcript
for reading accuracy, the same advantage Gemini gets in `--mode multimodal`.

In [14]:
YAML_INSTRUCTIONS = """
Output your response as a YAML list of entries. Each entry is a mapping with the
field names defined above. Use the exact field names from the prompt. Do not output
any prose, headers, markdown fences, or explanation -- only the raw YAML list.

Example output format:
- name: SMITH'S HOTEL
  address: 418 Johnson St
  city: Augusta
  state: GEORGIA
  category: HOTELS
- name: JONES BEAUTY PARLOR
  address: 122 Broad St
  city: Augusta
  state: GEORGIA
  category: BEAUTY PARLORS

If a field is missing or unknown, omit it entirely (do not write null or empty string).
Do not include any text before the first '-' or after the last entry.
"""

if ner_prompt_path.exists():
    base_prompt = ner_prompt_path.read_text(encoding='utf-8').strip()
    system_prompt = base_prompt + '\n' + YAML_INSTRUCTIONS
    print(f'NER prompt loaded ({len(base_prompt)} chars) + YAML instructions appended')
else:
    system_prompt = """You are extracting entries from a historical Green Book directory page.
Each entry describes a Black-owned or Black-friendly establishment.
Fields to extract: name, address, city, state, category.
""" + YAML_INSTRUCTIONS
    print('WARNING: ner_prompt.md not found -- using minimal fallback prompt')

print('\nSystem prompt preview (first 500 chars):')
print(system_prompt[:500])

NER prompt loaded (5552 chars) + YAML instructions appended

System prompt preview (first 500 chars):
You are a structured data extractor for a digitized version of "The Negro Motorist Green Book," a historical travel guide. Your goal is to identify and extract listings for businesses, schools, and services from the transcribed text, returning them as structured JSON.

## Source structure

This document is organized geographically and categorically. Entries are nested under state, city, and service category headings. Listings appear either as single-line entries (e.g., "Name—Address") or as larg


---
## Extract Entries

Runs the model on each page and collects raw YAML responses.
In `image_and_ocr` mode each user turn contains the page scan followed by the OCR transcript.
If no image is available for a page the model falls back to text-only automatically.

Adjust `EXTRACTION_MODE` and `MAX_PAGES` in the config cell to change behaviour.

In [15]:
import time


def extract_page(
    ocr_text: str,
    image_path: Path | None = None,
    max_new_tokens: int = 1024,
) -> str:
    """Run the model on one page; return raw YAML string."""
    use_image = image_path is not None and EXTRACTION_MODE != 'text_only'

    if use_image:
        img = load_image(image_path)  # PIL image, resized to MAX_IMAGE_PX
        if EXTRACTION_MODE == 'image_only':
            user_content = [
                {'type': 'image', 'image': img},
            ]
        else:  # image_and_ocr
            user_content = [
                {'type': 'image', 'image': img},
                {'type': 'text', 'text': f'OCR transcript (reference):\n\n{ocr_text.strip()}'},
            ]
    else:
        user_content = ocr_text.strip()

    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': user_content},
    ]

    # apply_chat_template with tokenize=True handles image encoding internally
    # enable_thinking=False suppresses <think>...</think> blocks (on by default in Qwen3.5)
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
        enable_thinking=False,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    # Decode only newly generated tokens
    return processor.decode(
        output_ids[0][inputs['input_ids'].shape[-1]:],
        skip_special_tokens=True,
    ).strip()


pages = ocr_files[:MAX_PAGES] if MAX_PAGES else ocr_files
raw_outputs: list[dict] = []

print(f'Processing {len(pages)} pages  [mode={EXTRACTION_MODE}] ...')
t0 = time.time()

for i, ocr_path in enumerate(pages, 1):
    ocr_text    = ocr_path.read_text(encoding='utf-8')
    img_path    = image_map.get(ocr_path)
    actual_mode = EXTRACTION_MODE if img_path else 'text_only'
    yaml_raw    = extract_page(ocr_text, image_path=img_path)
    raw_outputs.append({
        'page_name':   ocr_path.name,
        'image_path':  str(img_path) if img_path else None,
        'actual_mode': actual_mode,
        'ocr_text':    ocr_text,
        'yaml_raw':    yaml_raw,
    })
    elapsed = time.time() - t0
    print(f'  [{i}/{len(pages)}] {ocr_path.name}  [{actual_mode}]  ({elapsed:.0f}s elapsed)')

print(f'\nDone -- {len(raw_outputs)} pages in {time.time()-t0:.0f}s')
n_img = sum(1 for r in raw_outputs if r['image_path'])
print(f'Pages with image input: {n_img} / {len(raw_outputs)}')

Processing 10 pages  [mode=image_and_ocr] ...


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


KeyboardInterrupt: 

---
## Parse YAML -> Rows

In [ ]:
import yaml
import re

def parse_yaml_output(yaml_raw: str, page_name: str) -> tuple[list[dict], str | None]:
    """Parse YAML string into a list of entry dicts. Returns (entries, error_or_None)."""
    # Strip <think>...</think> blocks (Qwen3.5 thinking mode, in case enable_thinking=False didn't work)
    cleaned = re.sub(r'<think>[\s\S]*?</think>', '', yaml_raw, flags=re.IGNORECASE).strip()
    # Strip markdown fences
    cleaned = re.sub(r'^```[\w]*\n?|\n?```$', '', cleaned, flags=re.MULTILINE).strip()
    if not cleaned:
        return [], 'empty output'
    try:
        parsed = yaml.safe_load(cleaned)
    except yaml.YAMLError as e:
        return [], str(e)
    if parsed is None:
        return [], 'null result'
    if isinstance(parsed, dict):
        for key in ('entries', 'results', 'items'):
            if key in parsed and isinstance(parsed[key], list):
                parsed = parsed[key]
                break
        else:
            parsed = [parsed]
    if not isinstance(parsed, list):
        return [], f'unexpected type: {type(parsed)}'
    entries = []
    for item in parsed:
        if isinstance(item, dict):
            item['_page'] = page_name
            entries.append(item)
    return entries, None


all_entries: list[dict] = []
parse_errors: list[dict] = []

for result in raw_outputs:
    entries, error = parse_yaml_output(result['yaml_raw'], result['page_name'])
    if error:
        parse_errors.append({'page': result['page_name'], 'error': error, 'raw': result['yaml_raw'][:200]})
    all_entries.extend(entries)

print(f'Total entries parsed: {len(all_entries)}')
print(f'Parse errors:         {len(parse_errors)} / {len(raw_outputs)} pages')
if parse_errors:
    print('\nFailed pages:')
    for e in parse_errors:
        print(f"  {e['page']}: {e['error']}")
        print(f"  Raw (first 200): {e['raw']}")

---
## Review Extracted Entries

In [ ]:
import pandas as pd

df = pd.DataFrame(all_entries)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print()
df.head(20)

In [ ]:
key_cols = [c for c in df.columns if not c.startswith('_')]
fill = (df[key_cols].replace('', None).notna().mean() * 100).round(1)
print('Field fill rates:')
print(fill.to_string())

if 'state' in df.columns:
    print('\nTop states:')
    print(df['state'].value_counts().head(10).to_string())

if 'category' in df.columns:
    print('\nTop categories:')
    print(df['category'].value_counts().head(15).to_string())

In [ ]:
# Inspect raw model output for a specific page and display the source image
PAGE_INDEX = 0  # change this to inspect other pages

r = raw_outputs[PAGE_INDEX]
print(f'=== {r["page_name"]} ===')
print(f'Mode:  {r["actual_mode"]}')
print(f'Image: {r["image_path"] or "not found"}')
print('\n--- OCR input (first 800 chars) ---')
print(r['ocr_text'][:800])
print('\n--- Model YAML output ---')
print(r['yaml_raw'])

# Display the source page image inline
if r['image_path']:
    from IPython.display import display as ipy_display
    img = PILImage.open(r['image_path'])
    w, h = img.size
    if w > 600:
        img = img.resize((600, int(h * 600 / w)))
    ipy_display(img)

---
## Compare Against Existing Gemini Extraction

Side-by-side: how many entries did each model find per page, and how do the field fill rates compare?

In [ ]:
gemini_csvs = sorted(Path(PIPELINE_DIR, VOLUME).glob('entries_gemini-*.csv'))
if not gemini_csvs:
    print('No Gemini CSV found in volume directory -- skipping comparison')
else:
    gemini_path = gemini_csvs[-1]
    gemini_df   = pd.read_csv(gemini_path, dtype=str).fillna('')
    model_short = MODEL_ID.split('/')[-1]
    print(f'Gemini CSV:   {gemini_path.name}  ({len(gemini_df)} rows)')
    print(f'{model_short}: {len(df)} rows  (mode={EXTRACTION_MODE}, pages={len(pages)})')
    print()

    if 'canvas_fragment' in gemini_df.columns:
        import re as _re
        gemini_df['_image_id'] = gemini_df['canvas_fragment'].str.extract(r'/iiif/3/(\d+)/')
        gemini_counts = gemini_df.groupby('_image_id').size().rename('gemini')

        def _img_id(page_name):
            m = _re.search(r'\d{4}_(\d+)_', page_name)
            return m.group(1) if m else None

        df['_image_id'] = df['_page'].apply(_img_id)
        qwen_by_img = df.groupby('_image_id').size().rename('qwen')

        comparison = pd.concat([qwen_by_img, gemini_counts], axis=1).dropna()
        comparison['diff'] = comparison['qwen'] - comparison['gemini']
        print('Per-page entry counts (pages seen by both):')
        print(comparison.to_string())
        print(f'\nMean Qwen:   {comparison["qwen"].mean():.1f} entries/page')
        print(f'Mean Gemini: {comparison["gemini"].mean():.1f} entries/page')
        print(f'Mean diff:   {comparison["diff"].mean():+.1f} entries/page')
    else:
        print(f'Gemini: {len(gemini_df)} total  |  Qwen: {len(df)} total  (no per-page breakdown available)')

---
## Save Results

In [ ]:
model_slug = MODEL_ID.split('/')[-1].lower().replace('.', '-')
mode_slug  = EXTRACTION_MODE.replace('_', '-')
out_path   = Path(PIPELINE_DIR, VOLUME, f'entries_{model_slug}_{mode_slug}_yaml.csv')

save_cols = [c for c in df.columns if not c.startswith('_')]
df[save_cols].to_csv(out_path, index=False)
print(f'Saved {len(df)} entries -> {out_path}')